# RL-Chess-Engine: three seeds, measured against each run's starting network

The 2026-07-14 run had one training seed per arm, and evaluated each network with the search it
trained under. This notebook reruns the **baseline** and the **material-assisted** arm (the 10×
arm is skipped: it is most of the GPU time and showed no measurable change) for **three seeds**,
and for every trained network records:

- Elo vs a random mover, with and without the material blend in its search;
- Elo vs a one-ply material-greedy opponent (the network alone);
- the **starting network** of each seed under the same evaluations, so each arm is compared with
  where it began;
- a value-head probe: does it score "up a queen" above "down a queen"?

It also uses the **corrected draw rule** (2026-11-14): a game is drawn by repetition only when a
position has actually occurred three times, not as soon as a repeating move is available, which is
how the July run ended games. One extra run, seed 1's baseline under the old rule
(`--legacy-draw-claims`), measures how much that rule alone contributed to the draw cycle.

It also exports real self-play games and a web model from seed 1's assisted network (fixed in
advance, not picked by result) for the site.

**Kaggle setup:** Settings → Accelerator → **GPU T4 x2**; Settings → Internet → **On**; then
**Save Version → Save & Run All (Commit)**. Expect about 9 hours (Kaggle allows 12). No secrets are needed: results are
zipped to `/kaggle/working/chess_seeds_results.zip`; download it from the version's **Output** tab.

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/Danny-397/RL-Chess-Engine.git
%cd RL-Chess-Engine
!pip install -q python-chess onnx onnxscript
!git log --oneline -1

## 2 · Train baseline + assisted, three seeds (about 2.5 h per seed)

In [ ]:
SEEDS = [1, 2, 3]
for s in SEEDS:
    !python run_experiment.py --device cuda --skip-scaled --seed {s}
# The old draw rule, same seed, baseline only: how much did the rule itself cause?
!python run_experiment.py --device cuda --skip-scaled --skip-assisted --seed 1 --legacy-draw-claims

## 3 · Starting networks, same evaluations (CPU, runs in parallel)

In [ ]:
!python tools/no_learning_control.py --seed-values 1,2,3 --games 100 --workers 4 --opponent random --out experiment_out/start_random.json
!python tools/no_learning_control.py --seed-values 1,2,3 --games 100 --workers 4 --opponent greedy --out experiment_out/start_greedy.json

## 4 · Value-head probe on every network

In [ ]:
import glob
cks = sorted(glob.glob('experiment_out/seed*/baseline.pt') + glob.glob('experiment_out/seed*/assisted.pt'))
args = ' '.join(f'--checkpoint {c}' for c in cks)
!python tools/value_probe.py {args} --out experiment_out/value_probe.json

## 5 · Summary: each arm against its own starting network

In [ ]:
import json, math, sys
sys.path.insert(0, '.')
from evaluation import MatchResult

def se(m):
    lo, hi = m.elo_interval(); return (hi - lo) / (2 * 1.96)

start = {'random': json.load(open('experiment_out/start_random.json')),
         'greedy': json.load(open('experiment_out/start_greedy.json'))}
def start_cell(opp, w, seed):
    arm = next(a for a in start[opp]['arms'] if abs(a['material_weight'] - w) < 1e-9)
    c = next(p for p in arm['per_seed'] if p['seed'] == seed)
    return MatchResult(c['wins'], c['draws'], c['losses'], [])

rows = []
for s in SEEDS:
    for arm in ('baseline', 'assisted'):
        r = json.load(open(f'experiment_out/seed{s}/{arm}.result.json'))
        w = r['material_weight']
        checks = [
            ('random, own search', MatchResult(r['eval_wins'], r['eval_draws'], r['eval_losses'], []), start_cell('random', w, s)),
            ('random, w=0',        MatchResult(r['pure_eval_wins'], r['pure_eval_draws'], r['pure_eval_losses'], []), start_cell('random', 0.0, s)),
            ('greedy, w=0',        MatchResult(r['greedy_eval_wins'], r['greedy_eval_draws'], r['greedy_eval_losses'], []), start_cell('greedy', 0.0, s)),
        ]
        for label, trained, st in checks:
            d = trained.elo_difference - st.elo_difference
            e = math.hypot(se(trained), se(st))
            rows.append(dict(seed=s, arm=arm, eval=label, trained=round(trained.elo_difference),
                             start=round(st.elo_difference), change=round(d),
                             ci=[round(d - 1.96 * e), round(d + 1.96 * e)]))
for x in rows:
    print(f"seed {x['seed']}  {x['arm']:<9} {x['eval']:<19} trained {x['trained']:+5d}  start {x['start']:+5d}  change {x['change']:+5d}  95% CI {x['ci']}")
json.dump(rows, open('experiment_out/summary_vs_start.json', 'w'), indent=1)

# Draw rule: the same seed-1 baseline, old rule vs corrected rule (self-play termination).
import csv
for tag, d in (('corrected rule', 'experiment_out/seed1'), ('old rule', 'experiment_out/seed1-legacy-draws')):
    rows_t = list(csv.DictReader(open(f'{d}/term_baseline.csv')))
    print(tag, [(r.get('iteration'), r.get('non_decisive_rate')) for r in rows_t][-5:])

## 6 · For the site: real self-play games and a web model (seed 1, assisted; fixed in advance)

In [ ]:
!python tools/export_selfplay_games.py --checkpoint experiment_out/seed1/assisted.pt --label "GPU run, seed 1, material-assisted arm" --games 6 --out experiment_out/selfplay_games.json
!python export_web_model.py --checkpoint experiment_out/seed1/assisted.pt
!cp web/static/model.onnx web/static/model_meta.json experiment_out/

## 7 · Package everything

In [ ]:
!cd experiment_out && zip -qr /kaggle/working/chess_seeds_results.zip . -x 'checkpoints/*'
!ls -la /kaggle/working/